In [ ]:
import pandas as pd
import os

for csv_file in [file for file in os.listdir('data') if file.endswith('.csv')]:
    file_path = os.path.join('data', csv_file)
    df = pd.read_csv(file_path)
    print(f"Columns in {csv_file}:")
    print(df.columns.tolist())
    print()

In [ ]:
import pandas as pd
import os

# Path to your data folder
data_folder = 'data'

# List all CSV files in the folder
csv_files = [file for file in os.listdir(data_folder) if file.endswith('.csv')]

# Create an empty list to store DataFrames for each year
dfs = []

# Standardize the column mapping
column_mapping = {
    'Country or region': 'Country',
    'Happiness.Score': 'Happiness Score',
    'Score': 'Happiness Score',
    'Happiness.Rank': 'Happiness Rank',
    'Overall rank': 'Happiness Rank',
    'Economy (GDP per Capita)': 'GDP per Capita',
    'Economy..GDP.per.Capita.': 'GDP per Capita',
    'GDP per capita': 'GDP per Capita',
    'Health (Life Expectancy)': 'Life Expectancy',
    'Health..Life.Expectancy.': 'Life Expectancy',
    'Healthy life expectancy': 'Life Expectancy',
    'Freedom to make life choices' : 'Freedom',
    'Trust (Government Corruption)': 'Trust in Government',
    'Trust..Government.Corruption.': 'Trust in Government',
    'Perceptions of corruption': 'Trust in Government',
    'Dystopia.Residual': 'Dystopia Residual'
}

country_mapping = {
'Hong Kong S.A.R., China': 'Hong Kong',
'Macedonia': 'North Macedonia',
'North Cyprus': 'Northern Cyprus',
'Somaliland region': 'Somaliland Region',
'Taiwan Province of China': 'Taiwan',
'Trinidad & Tobago': 'Trinidad and Tobago'
}

# Process each CSV file
for csv_file in csv_files:
    # Extract year from the file name
    year = int(csv_file.split('.')[0])

    file_path = os.path.join(data_folder, csv_file)
    df = pd.read_csv(file_path)
    
    # Rename columns based on the mapping
    df.rename(columns=column_mapping, inplace=True)

    # Apply country name mapping
    df['Country'] = df['Country'].replace(country_mapping)
    
    # Add the 'Year' column
    df['Year'] = year
    
    # Append to the list of DataFrames
    dfs.append(df)

# Concatenate all DataFrames into one
merged_df = pd.concat(dfs, ignore_index=True)

# Create a dictionary for country-region mapping from existing data
region_mapping = dict(
    merged_df[merged_df['Year'].isin([2015, 2016])]
    [['Country', 'Region']].drop_duplicates().values
)

# Apply mapping to populate the 'Region' column
merged_df['Region'] = merged_df['Country'].map(region_mapping)

# Assign 'Sub-Saharan Africa' as the region for Gambia
merged_df.loc[merged_df['Country'] == 'Gambia', 'Region'] = 'Sub-Saharan Africa'

In [ ]:
min_value = merged_df['Year'].min()
max_value = merged_df['Year'].max()
print(min_value)
print(max_value)

merged_df

In [ ]:
# Create a DataFrame with unique countries and their corresponding years
unique_countries = merged_df[['Country', 'Year']].drop_duplicates()

# Sort by country and year to view data neatly
unique_countries_sorted = unique_countries.sort_values(by=['Country', 'Year'])

# Optional: Pivot the table to view which years each country appears in
country_year_pivot = unique_countries_sorted.pivot_table(index='Country', columns='Year', aggfunc='size', fill_value=0)

# Display the unique countries and the pivot table
# print("Unique country entries with their appearance years:")
# print(unique_countries_sorted)
print("\nCountry-Year Appearance Table:")
print(country_year_pivot)

In [ ]:
# Create a dictionary for country-region mapping from existing data
region_mapping = dict(
    merged_df[merged_df['Year'].isin(['2015', '2016'])]
    [['Country', 'Region']].drop_duplicates().values
)

# Apply mapping to populate the 'Region' column
merged_df['Region'] = merged_df['Country'].map(region_mapping)

# Identify countries still missing region information
missing_regions = merged_df[merged_df['Region'].isnull()][['Country']].drop_duplicates()

print("Countries still missing region information:")
print(missing_regions)

In [ ]:
# Pivoting - keep the relevant columns
pivoted_df = pd.pivot_table(
    merged_df,
    index='Country',
    columns='Year',
    values=['Region', 'Happiness Score', 'Happiness Rank', 'GDP per Capita', 'Life Expectancy', 'Freedom', 'Corruption', 'Generosity'],
    aggfunc='first'
)

pivoted_df.columns = [f'{metric}_{year}' for metric, year in pivoted_df.columns]
pivoted_df.reset_index(inplace=True)

pivoted_df

In [6]:
# Filter DataFrame to include only numeric columns
numeric_cols = merged_df.select_dtypes(include='number').columns.tolist()
if 'Year' in numeric_cols:
    numeric_cols.remove('Year')

# Group by Region and Year, only taking the mean of numeric columns
region_agg_df = merged_df.groupby(['Region', 'Year'])[numeric_cols].mean().reset_index()

# Examine the processed DataFrame
print(region_agg_df.head())

                      Region  Year  Happiness Rank  Happiness Score  \
0  Australia and New Zealand  2015             9.5           7.2850   
1  Australia and New Zealand  2016             8.5           7.3235   
2  Australia and New Zealand  2017             9.0           7.2990   
3  Australia and New Zealand  2018             9.0           7.2980   
4  Australia and New Zealand  2019             9.5           7.2675   

   Standard Error  GDP per Capita    Family  Life Expectancy   Freedom  \
0         0.03727        1.291880  1.314450         0.919965  0.645310   
1             NaN        1.402545  1.138770         0.841080  0.574920   
2             NaN        1.445060  1.529119         0.830323  0.607835   
3             NaN        1.304000       NaN         0.893000  0.658000   
4             NaN        1.337500       NaN         1.031000  0.571000   

   Corruption  Generosity  Dystopia Residual  Lower Confidence Interval  \
0    0.392795    0.455315           2.265355         

In [10]:
year_data = merged_df[merged_df['Year'] == 2019]
rank_df = year_data.sort_values('Happiness Rank', ascending=True).head(10)
rank_df

,Country,Region,Happiness Rank,Happiness Score,Standard Error,GDP per Capita,Family,Life Expectancy,Freedom,Corruption,Generosity,Dystopia Residual,Year,Lower Confidence Interval,Upper Confidence Interval,Whisker.high,Whisker.low,Social support
626,Finland,Western Europe,1,7.769,NaN,1.340,NaN,0.986,0.596,0.393,0.153,NaN,2019,NaN,NaN,NaN,NaN,1.587
627,Denmark,Western Europe,2,7.600,NaN,1.383,NaN,0.996,0.592,0.410,0.252,NaN,2019,NaN,NaN,NaN,NaN,1.573
628,Norway,Western Europe,3,7.554,NaN,1.488,NaN,1.028,0.603,0.341,0.271,NaN,2019,NaN,NaN,NaN,NaN,1.582
629,Iceland,Western Europe,4,7.494,NaN,1.380,NaN,1.026,0.591,0.118,0.354,NaN,2019,NaN,NaN,NaN,NaN,1.624
630,Netherlands,Western Europe,5,7.488,NaN,1.396,NaN,0.999,0.557,0.298,0.322,NaN,2019,NaN,NaN,NaN,NaN,1.522
631,Switzerland,Western Europe,6,7.480,NaN,1.452,NaN,1.052,0.572,0.343,0.263,NaN,2019,NaN,NaN,NaN,NaN,1.526
632,Sweden,Western Europe,7,7.343,NaN,1.387,NaN,1.009,0.574,0.373,0.267,NaN,2019,NaN,NaN,NaN,NaN,1.487
633,New Zealand,Australia and New Zealand,8,7.307,NaN,1.303,NaN,1.026,0.585,0.380,0.330,NaN,2019,NaN,NaN,NaN,NaN,1.557
634,Canada,North America,9,7.278,NaN,1.365,NaN,1.039,0.584,0.308,0.285,NaN,2019,NaN,NaN,NaN,NaN,1.505
635,Austria,Western Europe,10,7.246,NaN,1.376,NaN,1.016,0.532,0.226,0.244,NaN,2019,NaN,NaN,NaN,NaN,1.475
